In [15]:
# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported")

Libraries Imported


In [17]:
# =========================================================
# LOAD DATA
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train.csv'
test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)

Train Shape : (77299, 11)
Test Shape  : (41778, 10)


In [18]:
# =========================================================
# CREATE COPIES
# =========================================================

train_df = train.copy()
test_df = test.copy()

print("Copies Created")

Copies Created


In [19]:
# =========================================================
# CONVERT TIMESTAMP
# =========================================================

# Assume a base date for the 'day' column, e.g., '1970-01-01'
base_date = pd.to_datetime('1970-01-01')

# For train_df
train_date_part = base_date + pd.to_timedelta(train_df['day'], unit='D')
train_df['timestamp'] = pd.to_datetime(train_date_part.dt.strftime('%Y-%m-%d') + ' ' + train_df['timestamp'])

# For test_df
test_date_part = base_date + pd.to_timedelta(test_df['day'], unit='D')
test_df['timestamp'] = pd.to_datetime(test_date_part.dt.strftime('%Y-%m-%d') + ' ' + test_df['timestamp'])

print("Timestamp Converted")

Timestamp Converted


In [20]:
# =========================================================
# COMBINE TRAIN + TEST
# =========================================================

combined = pd.concat([train_df, test_df], axis=0)

print("Combined Shape :", combined.shape)

Combined Shape : (119077, 11)


In [21]:
# =========================================================
# EXTRACT TIME FEATURES
# =========================================================

combined['hour'] = combined['timestamp'].dt.hour
combined['dayofweek'] = combined['timestamp'].dt.dayofweek

print("Time Features Extracted")

Time Features Extracted


In [22]:
# =========================================================
# MISSING VALUES BEFORE IMPUTATION
# =========================================================

combined.isnull().sum()

,0
Index,0
geohash,0
day,0
timestamp,0
demand,41778
RoadType,924
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,3844


In [23]:
# =========================================================
# CREATE MISSING INDICATORS
# =========================================================

missing_cols = ['Temperature', 'Weather', 'RoadType']

for col in missing_cols:

    combined[col + '_missing'] = combined[col].isnull().astype(int)

print("Missing Indicators Created")

Missing Indicators Created


In [24]:
# =========================================================
# HELPER FUNCTION
# =========================================================

def fill_mode(series):

    mode = series.mode()

    if len(mode) > 0:
        return series.fillna(mode[0])

    return series

In [25]:
# =========================================================
# TEMPERATURE IMPUTATION - LEVEL 1
# =========================================================

combined['Temperature'] = combined.groupby(
    ['geohash', 'Weather', 'hour']
)['Temperature'].transform(
    lambda x: x.fillna(x.median())
)

print("Temperature Level 1 Done")

Temperature Level 1 Done


In [26]:
# =========================================================
# TEMPERATURE IMPUTATION - LEVEL 2
# =========================================================

combined['Temperature'] = combined.groupby(
    ['geohash', 'hour']
)['Temperature'].transform(
    lambda x: x.fillna(x.median())
)

print("Temperature Level 2 Done")

Temperature Level 2 Done


In [27]:
# =========================================================
# TEMPERATURE IMPUTATION - LEVEL 3
# =========================================================

combined['Temperature'] = combined.groupby(
    ['Weather', 'hour']
)['Temperature'].transform(
    lambda x: x.fillna(x.median())
)

print("Temperature Level 3 Done")

Temperature Level 3 Done


In [28]:
# =========================================================
# FINAL TEMPERATURE FALLBACK
# =========================================================

combined['Temperature'].fillna(
    combined['Temperature'].median(),
    inplace=True
)

print("Final Temperature Fallback Done")

Final Temperature Fallback Done


In [29]:
# =========================================================
# WEATHER IMPUTATION - LEVEL 1
# =========================================================

combined['Weather'] = combined.groupby(
    ['geohash', 'hour']
)['Weather'].transform(fill_mode)

print("Weather Level 1 Done")

Weather Level 1 Done


In [30]:
# =========================================================
# CREATE TEMPERATURE BUCKETS
# =========================================================

combined['temp_bucket'] = pd.cut(
    combined['Temperature'],
    bins=10
)

print("Temperature Buckets Created")

Temperature Buckets Created


In [31]:
# =========================================================
# WEATHER IMPUTATION - LEVEL 2
# =========================================================

combined['Weather'] = combined.groupby(
    ['geohash', 'temp_bucket']
)['Weather'].transform(fill_mode)

print("Weather Level 2 Done")

Weather Level 2 Done


In [32]:
# =========================================================
# FINAL WEATHER FALLBACK
# =========================================================

combined['Weather'].fillna(
    combined['Weather'].mode()[0],
    inplace=True
)

print("Final Weather Fallback Done")

Final Weather Fallback Done


In [33]:
# =========================================================
# ROADTYPE IMPUTATION - LEVEL 1
# geohash + NumberofLanes + LargeVehicles
# =========================================================

combined['RoadType'] = combined.groupby(
    ['geohash', 'NumberofLanes', 'LargeVehicles']
)['RoadType'].transform(fill_mode)

print("RoadType Level 1 Done")

RoadType Level 1 Done


In [34]:
# =========================================================
# ROADTYPE IMPUTATION - LEVEL 2
# geohash + NumberofLanes
# =========================================================

combined['RoadType'] = combined.groupby(
    ['geohash', 'NumberofLanes']
)['RoadType'].transform(fill_mode)

print("RoadType Level 2 Done")

RoadType Level 2 Done


In [35]:
# =========================================================
# ROADTYPE IMPUTATION - LEVEL 3
# NumberofLanes + LargeVehicles
# =========================================================

combined['RoadType'] = combined.groupby(
    ['NumberofLanes', 'LargeVehicles']
)['RoadType'].transform(fill_mode)

print("RoadType Level 3 Done")

RoadType Level 3 Done


In [36]:
# =========================================================
# ROADTYPE IMPUTATION - LEVEL 4
# geohash
# =========================================================

combined['RoadType'] = combined.groupby(
    ['geohash']
)['RoadType'].transform(fill_mode)

print("RoadType Level 4 Done")

RoadType Level 4 Done


In [37]:
# =========================================================
# FINAL ROADTYPE FALLBACK
# =========================================================

combined['RoadType'].fillna(
    combined['RoadType'].mode()[0],
    inplace=True
)

print("Final RoadType Fallback Done")

Final RoadType Fallback Done


In [38]:
# =========================================================
# CHECK MISSING VALUES AFTER IMPUTATION
# =========================================================

combined.isnull().sum()

,0
Index,0
geohash,0
day,0
timestamp,0
demand,41778
RoadType,0
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,0


In [39]:
# =========================================================
# SPLIT BACK
# =========================================================

train_df = combined.iloc[:len(train)].copy()
test_df = combined.iloc[len(train):].copy()

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

Train Shape : (77299, 17)
Test Shape  : (41778, 17)


In [40]:
# =========================================================
# DROP TEMPORARY COLUMN
# =========================================================

train_df.drop(columns=['temp_bucket'], inplace=True)
test_df.drop(columns=['temp_bucket'], inplace=True)

print("Temporary Columns Removed")

Temporary Columns Removed


In [41]:
# =========================================================
# FINAL CHECK
# =========================================================

print(train_df.isnull().sum())

print("\n")

print(test_df.isnull().sum())

Index                  0
geohash                0
day                    0
timestamp              0
demand                 0
RoadType               0
NumberofLanes          0
LargeVehicles          0
Landmarks              0
Temperature            0
Weather                0
hour                   0
dayofweek              0
Temperature_missing    0
Weather_missing        0
RoadType_missing       0
dtype: int64


Index                      0
geohash                    0
day                        0
timestamp                  0
demand                 41778
RoadType                   0
NumberofLanes              0
LargeVehicles              0
Landmarks                  0
Temperature                0
Weather                    0
hour                       0
dayofweek                  0
Temperature_missing        0
Weather_missing            0
RoadType_missing           0
dtype: int64


In [43]:
# =========================================================
# SAVE IMPUTED DATASETS
# =========================================================

train_save_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_imputed.csv'
test_save_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_imputed.csv'

train_df.to_csv(train_save_path, index=False)
test_df.to_csv(test_save_path, index=False)

print("Imputed datasets saved successfully")

Imputed datasets saved successfully
